In [ ]:
#!/usr/bin/env python
# coding: utf-8

import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualBlock2D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.norm1 = nn.InstanceNorm2d(in_channels)
        self.conv_a = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.relu = nn.LeakyReLU(0.2, inplace=False)
        self.conv_b1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.norm2 = nn.InstanceNorm2d(out_channels)
        self.conv_b2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)

    def forward(self, x):
        x = self.norm1(x)
        a = self.conv_a(x)
        b = self.relu(x)
        b = self.conv_b1(b)
        b = self.norm2(b)
        b = self.relu(b)
        b = self.conv_b2(b)
        return a + b

# --- Region-Aware Global FiLM ---
class RegionAwareFiLM(nn.Module):
    def __init__(self, num_channels):
        super().__init__()
        # small MLP that transforms [mean,std] -> gamma/beta per channel
        self.mlp = nn.Sequential(
            nn.Conv2d(2 * num_channels, num_channels, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(num_channels, 2 * num_channels, kernel_size=1)
        )
        # register buffers sized per-channel to avoid shape mismatch when loading checkpoints
        self.register_buffer("last_mean", torch.zeros(num_channels))
        self.register_buffer("last_std", torch.zeros(num_channels))

    def forward(self, x, noise_map=None):
        mean = F.adaptive_avg_pool2d(x, 1)   
        std = torch.sqrt(F.adaptive_avg_pool2d((x - mean) ** 2, 1) + 1e-5) 
        with torch.no_grad():
            m = mean.view(mean.shape[0], mean.shape[1])
            s = std.view(std.shape[0], std.shape[1])

            self.last_mean = m.mean(dim=0).cpu()
            self.last_std = s.mean(dim=0).cpu()

        stats = torch.cat([mean, std], dim=1)  # (B,2C,1,1)
        gamma_beta = self.mlp(stats)           # (B,2C,1,1)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=1)
        out = gamma * x + beta
        return out


# --- Residual Block with Global FiLM (encoder) ---
class ResidualBlock2D_GlobalFiLM(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.norm1 = nn.InstanceNorm2d(out_channels, affine=True)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.norm2 = nn.InstanceNorm2d(out_channels, affine=True)
        self.film = RegionAwareFiLM(out_channels)
        self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x, noise_map=None):
        identity = self.shortcut(x)
        out = self.conv1(x)
        out = self.norm1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.norm2(out)
        out = self.film(out, noise_map)
        out = torch.nan_to_num(out, nan=0.0, posinf=1.0, neginf=0.0)
        return self.relu(out + identity)
# --- Local FiLM ---
class LocalFiLM(nn.Module):
    def __init__(self, feat_channels, num_regions=3, hidden_channels=None):
        super().__init__()
        hidden = hidden_channels or max(64, feat_channels // 2)
        self.condition = nn.Sequential(
            nn.Conv2d(feat_channels + num_regions, hidden, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, 2 * feat_channels, kernel_size=1)
        )

    def forward(self, features, region_maps):
        if region_maps is None:
            return features
        if region_maps.shape[2:] != features.shape[2:]:
            region_maps = F.interpolate(region_maps, size=features.shape[2:], mode='bilinear', align_corners=False)
        fused = torch.cat([features, region_maps], dim=1)
        gamma_beta = self.condition(fused)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=1)
        return gamma * features + beta

    def extract_region_conditioned_gamma_beta(self, features, region_maps):
        B, C, H, W = features.shape
    
        if region_maps is None:
            print("  region_maps is None")
            return None, None
    
    
        if region_maps.shape[2:] != (H, W):
            region_maps = F.interpolate(region_maps, size=(H, W), mode='bilinear', align_corners=False)
    
        try:
            fused = torch.cat([features, region_maps], dim=1)
        except Exception as e:
            print(f"[ERROR] torch.cat failed with error: {e}")
            print(f"  features shape: {features.shape}")
            print(f"  region_maps shape: {region_maps.shape}")
            raise
    
        gamma_beta = self.condition(fused)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=1)
        return gamma.detach(), beta.detach()




# --- Residual Block with Local FiLM ---
class ResidualBlock2D_LocalFiLM(nn.Module):
    def __init__(self, in_channels, out_channels, num_regions=3):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.norm1 = nn.InstanceNorm2d(out_channels, affine=True)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.norm2 = nn.InstanceNorm2d(out_channels, affine=True)
        self.local_film = LocalFiLM(out_channels, num_regions=num_regions)
        self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x, region_maps):
        identity = self.shortcut(x)
        out = self.conv1(x)
        out = self.norm1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.norm2(out)
        out = self.local_film(out, region_maps)
        out = torch.nan_to_num(out, nan=0.0, posinf=1.0, neginf=0.0)
        return self.relu(out + identity)

    def pre_film_features(self, x):
        out = self.conv1(x)
        out = self.norm1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.norm2(out)
        return out

# --- Encoder ---
class HybridEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_in = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.pool = nn.AvgPool2d(kernel_size=2)
        self.rb1 = ResidualBlock2D_GlobalFiLM(32, 32)
        self.rb2 = ResidualBlock2D_GlobalFiLM(32, 64)
        self.rb3 = ResidualBlock2D_GlobalFiLM(64, 128)
        self.rb4 = ResidualBlock2D(128, 256)
        self.rb5 = ResidualBlock2D(256, 512)
        self.rb6 = ResidualBlock2D(512, 512)
        self.rb7 = ResidualBlock2D(512, 512)

    def forward(self, x, noise_map=None):
        x1 = self.rb1(self.conv_in(x), noise_map)
        x2 = self.rb2(self.pool(x1), noise_map)
        x3 = self.rb3(self.pool(x2), noise_map)
        x4 = self.rb4(self.pool(x3))
        x5 = self.rb5(self.pool(x4))
        x6 = self.rb6(self.pool(x5))
        x7 = self.rb7(self.pool(x6))
        return x7, [x6, x5, x4, x3, x2, x1]

# --- Decoder ---
class HybridDecoder(nn.Module):
    def __init__(self, num_regions=3):
        super().__init__()
        self.up1 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.res1 = ResidualBlock2D_LocalFiLM(1024, 512, num_regions=num_regions)
        self.up2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.res2 = ResidualBlock2D_LocalFiLM(1024, 256, num_regions=num_regions)
        self.up3 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.res3 = ResidualBlock2D_LocalFiLM(512, 128, num_regions=num_regions)
        self.up4 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.res4 = ResidualBlock2D(256, 64)

        self.up5 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.res5 = ResidualBlock2D(128, 32)

        self.up6 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.res6 = ResidualBlock2D(64, 64)

        self.final = ResidualBlock2D(64, 32)
        self.out_conv = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x, skips, region_maps):
        x6, x5, x4, x3, x2, x1 = skips
        x = self.up1(x)
        x = self.res1(torch.cat([x, x6], dim=1), region_maps)
        x = self.up2(x)
        x = self.res2(torch.cat([x, x5], dim=1), region_maps)
        x = self.up3(x)
        x = self.res3(torch.cat([x, x4], dim=1), region_maps)
        x = self.up4(x)
        x = self.res4(torch.cat([x, x3], dim=1))
        x = self.up5(x)
        x = self.res5(torch.cat([x, x2], dim=1))
        x = self.up6(x)
        x = self.res6(torch.cat([x, x1], dim=1))
        x = self.final(x)
        return torch.clamp(self.out_conv(x), 0.0, 1.0)

    def get_res1_pre_film(self, latent, skip6):
        x_up = self.up1(latent)
        return self.res1.pre_film_features(torch.cat([x_up, skip6], dim=1))

# --- Full UNet ---
class DenoisingUNet2D_HybridFiLM(nn.Module):
    def __init__(self, num_regions=3):
        super().__init__()
        self.encoder = HybridEncoder()
        self.decoder = HybridDecoder(num_regions=num_regions)

    def forward(self, x, noise_map=None, region_maps=None):
        latent, skips = self.encoder(x, noise_map)
        return self.decoder(latent, skips, region_maps)
